### Reading Silver Table

In [0]:
%sql
use catalog merchai_data;

In [0]:
from pyspark.sql.functions import *

silver_df = spark.table("silver.silver_customer_orders")


In [0]:
from pyspark.sql.functions import col

silver_df = silver_df.filter(
    col("product_name").isNotNull() &
    col("department").isNotNull() &
    col("aisle").isNotNull()
)

In [0]:
silver_df.printSchema()

### Building Gold Layer

##### Gold Customer Features (Characteristics of each customer)

In [0]:
customer_features = (
    silver_df
    .groupBy("user_id")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        count("product_id").alias("total_products"),
        round(avg("reordered"), 2).alias("reorder_rate"),
        round(count("product_id") / countDistinct("order_id"), 2).alias("avg_basket_size"),
        round(avg("days_since_prior_order"), 2).alias("avg_days_between_orders"),
        sum("reordered").alias("total_reorders")
    )
)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import desc, row_number

department_counts = (
    silver_df
    .groupBy("user_id", "department")
    .count()
)

department_window = Window.partitionBy("user_id").orderBy(desc("count"))

favorite_department = (
    department_counts
    .withColumn("rank", row_number().over(department_window))
    .filter(col("rank") == 1)
    .select(
        "user_id",
        col("department").alias("favorite_department")
    )
)

In [0]:
aisle_counts = (
    silver_df
    .groupBy("user_id", "aisle")
    .count()
)

aisle_window = Window.partitionBy("user_id").orderBy(desc("count"))

favorite_aisle = (
    aisle_counts
    .withColumn("rank", row_number().over(aisle_window))
    .filter(col("rank") == 1)
    .select(
        "user_id",
        col("aisle").alias("favorite_aisle")
    )
)

In [0]:
gold_customer_features = (
    customer_features
    .join(favorite_department, "user_id", "left")
    .join(favorite_aisle, "user_id", "left")
)


In [0]:
display(gold_customer_features.limit(5))

In [0]:
gold_customer_features.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("merchai_data.gold.gold_customer_features")

##### Gold Product Metrics (Most popular and frequently reordered Product)

In [0]:
gold_product_metrics = (
    silver_df
    .groupBy("product_id", "product_name")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        round(avg("reordered"), 2).alias("reorder_rate"),
        countDistinct("user_id").alias("unique_customers")
    )
)

display(gold_product_metrics.limit(5))

In [0]:
gold_product_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("merchai_data.gold.gold_product_metrics")

##### Gold Department Metrics (Each Department Performance)

In [0]:
gold_department_metrics = (
    silver_df
    .groupBy("department")
    .agg(
        countDistinct("product_id").alias("total_products"),
        countDistinct("order_id").alias("total_orders")
    )
)

display(gold_department_metrics.limit(50))

In [0]:
gold_department_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("merchai_data.gold.gold_department_metrics")

In [0]:
tables = [
    "gold_customer_features",
    "gold_product_metrics",
    "gold_department_metrics"
]

for table in tables:
    df = spark.table(f"merchai_data.gold.{table}")
    print(f"{table}")
    print(f"Rows    : {df.count()}")
    print(f"Columns : {len(df.columns)}")
    print(f"Features: {df.columns}")
    print("-" * 80)